In [ ]:
import pandas as pd
import plotly.express as px
import numpy as np

# caminho = só o endereço do arquivo (string)
caminho_matches = '../data/processed/df_dados_matches.csv'
caminho_players = '../data/processed/df_dados_players.csv'
caminho_teams = '../data/processed/df_dados_teams.csv'

# df = o DataFrame carregado (tabela de dados)
df_matches = pd.read_csv(caminho_matches)
df_players = pd.read_csv(caminho_players)
df_teams = pd.read_csv(caminho_teams)

# Criando variável da média de gols por temporada
df_matches['TotalGols'] = df_matches['FullTimeHomeGoals'] + df_matches['FullTimeAwayGoals']
media_gols = df_matches.groupby('Season')['TotalGols'].mean().reset_index()

# Criando o gráfico de média de gols por temporada
grafico_media = px.line(media_gols, x='Season', y="TotalGols", labels={"TotalGols": "Média de gols", "Season": "Temporada"}) 
grafico_media.update_layout(title="Média de gols por temporada", title_x=0.5, xaxis_tickangle=-45)

# Criando colunas de pontos para time da casa e fora de casa
df_matches['PtsHome'] = df_matches['FullTimeResult'].map({'H': 3, 'D': 1, 'A': 0})
df_matches['PtsAway'] = df_matches['FullTimeResult'].map({'H': 0, 'D': 1, 'A': 3})

# Criando variáveis de pontos em casa e fora e resetando index
pts_casa = df_matches.groupby(['HomeTeam', 'Season'])['PtsHome'].sum().reset_index()
pts_fora = df_matches.groupby(['AwayTeam', 'Season'])['PtsAway'].sum().reset_index()

# Renomeando colunas das variáveis de pontos
pts_casa.columns = ['Team', 'Season', 'Pts'] 
pts_fora.columns = ['Team', 'Season', 'Pts']

# Criando df de pontos totais dos times em cada temporada
df_pontos = pd.concat([pts_casa, pts_fora], ignore_index=True)
pts_total = df_pontos.groupby(['Team', 'Season'])['Pts'].sum().reset_index()

# Criando df dos campeões
df_campeoes = pts_total.loc[pts_total.groupby('Season')['Pts'].idxmax()]

# Criando gráfico dos campeões
n_campeoes = df_campeoes.value_counts('Team').reset_index()
grafico_campeoes = px.bar(n_campeoes, x='Team', y='count', labels={'count': 'Número de vezes', 'Team': 'Clube'})
grafico_campeoes.update_layout(title="Campeões da Premier League 2000-2025", title_x=0.5)

# Criando df dos rebaixados
df_rebaixados = pts_total.sort_values('Pts').groupby('Season').head(3)

# Criando gráfico dos rebaixados
n_rebaixados = df_rebaixados.value_counts('Team').reset_index()
grafico_rebaixados = px.bar(n_rebaixados.head(10), x='Team', y='count', labels={'count': 'Número de vezes', 'Team': 'Clube'})
grafico_rebaixados.update_layout(title='Top 10 mais rebaixados da Premier League 2000-2025', title_x=0.5)

# Criando comparação sem torcida contra com torcida jogando em casa
condicoes_torcida = [
    (df_matches['Season'] == '2019/20') | (df_matches['Season'] == '2020/21'),
    df_matches['Season'] > '2020/21',
    df_matches['Season'] < '2019/20'
]

valores_torcida = [
    'Pandemia (Sem torcida)',
    'Depois da pandemia',
    'Antes da pandemia'
]

# Criando df da torcida
df_matches['Periodo'] = np.select(condicoes_torcida, valores_torcida, default='')
df_torcida = df_matches.groupby('Periodo')['FullTimeResult'].apply(lambda x: (x == 'H').mean() * 100).reset_index(name='Pct_Vitorias_Casa') 

# Criando gráfico de comparação de vitórias com e sem torcida
grafico_torcida = px.bar(df_torcida, x='Periodo', y='Pct_Vitorias_Casa', labels={'Periodo': 'Período', 'Pct_Vitorias_Casa': 'Porcentagem de vitórias em casa'}, category_orders={"Periodo": ['Antes da pandemia', 'Pandemia (Sem torcida)', 'Depois da pandemia']})
grafico_torcida.update_layout(title='Comparação de vitória com e sem torcida jogando em casa', title_x=0.5)

# Power BI
df_campeoes = pts_total.loc[pts_total.groupby('Season')['Pts'].idxmax()]
titulos = df_campeoes['Team'].value_counts().reset_index()
titulos.columns = ['Team', 'Titulos']
titulos.to_csv('../data/processed/df_titulos.csv', index=False)
n_rebaixados.to_csv('../data/processed/df_rebaixados.csv', index=False)
df_torcida.to_csv('../data/processed/df_torcida.csv', index=False)
df_torcida['Pct_Vitorias_Casa'] = df_torcida['Pct_Vitorias_Casa'].round(1)
df_torcida.to_csv('../data/processed/df_torcida.csv', index=False)
df_torcida['Ordem'] = df_torcida['Periodo'].map({
    'Antes da pandemia': 1,
    'Pandemia (Sem torcida)': 2,
    'Depois da pandemia': 3
})
df_torcida.to_csv('../data/processed/df_torcida.csv', index=False)
